# PhotoMedGemma — Kaggle Validation Notebook

**Purpose:** Run Google MedGemma 4B-IT on a DICOM medical image, extract layer-by-layer activations, and compare to our photonic chip simulation results.

## Kaggle setup

**Secrets:** Add `HF_TOKEN` in Kaggle → Add-ons → Secrets  
**Accelerator:** GPU T4 (free tier works with 4-bit quantization)  
**Model access:** Accept terms at https://huggingface.co/google/medgemma-4b-it  

**DICOM sources (free):**
- [NIH Chest X-rays](https://www.kaggle.com/datasets/nih-chest-xrays/data)
- [RSNA Pneumonia Detection](https://www.kaggle.com/competitions/rsna-pneumonia-detection-challenge)
- [COVID-19 Radiography](https://www.kaggle.com/datasets/tawsifurrahman/covid19-radiography-database)

## Quantization modes

| Mode | VRAM | Use when |
|------|------|----------|
| `QUANTIZE_4BIT = True` | ~3.5GB | Free T4 GPU, inference + activation capture |
| `QUANTIZE_4BIT = False` | ~8GB | P100/A100, full precision weights for SVD |

**Note:** 4-bit quantized weights are dequantized to float32 before SVD compilation,
so the photonic simulation results are valid regardless of which mode you choose.

In [ ]:
!pip install -q pydicom transformers accelerate bitsandbytes
!pip install -q Pillow numpy scipy

In [ ]:
import os, json
import numpy as np
from pathlib import Path
from typing import Tuple

# ── Configuration ──────────────────────────────────────────────────────────
MODEL_ID       = 'google/medgemma-4b-it'
DICOM_PATH     = '/kaggle/input/your-dataset/scan.dcm'   # ← change this
SVD_RANK       = 64        # photonic chip SVD truncation rank
TARGET_LAYERS  = 2         # how many transformer layers to hook
QUANTIZE_4BIT  = True      # True = 4-bit (fits T4), False = bfloat16 (needs P100+)
OUTPUT_DIR     = Path('/kaggle/working/photomedgemma_outputs')
OUTPUT_DIR.mkdir(exist_ok=True)

# ── HuggingFace token ──────────────────────────────────────────────────────
try:
    from kaggle_secrets import UserSecretsClient
    HF_TOKEN = UserSecretsClient().get_secret('HF_TOKEN')
    print('HF token: loaded from Kaggle secrets')
except Exception:
    HF_TOKEN = os.environ.get('HF_TOKEN', '')
    print(f'HF token: from env ({bool(HF_TOKEN)})')

print(f'Model:        {MODEL_ID}')
print(f'Quantization: {"4-bit (bitsandbytes)" if QUANTIZE_4BIT else "bfloat16 full"}')
print(f'SVD rank:     {SVD_RANK}')
print(f'Output dir:   {OUTPUT_DIR}')

## 1. Load DICOM

In [ ]:
import pydicom
from PIL import Image

def load_dicom_as_pil(path: str) -> Tuple[Image.Image, dict]:
    ds = pydicom.dcmread(path)
    pixels = ds.pixel_array.astype(np.float32)
    if getattr(ds, 'PhotometricInterpretation', '') == 'MONOCHROME1':
        pixels = pixels.max() - pixels
    lo, hi = pixels.min(), pixels.max()
    if hi > lo:
        pixels = (pixels - lo) / (hi - lo) * 255.0
    pixels = pixels.clip(0, 255).astype(np.uint8)
    img = Image.fromarray(pixels if pixels.ndim == 3 else pixels, mode='L' if pixels.ndim == 2 else 'RGB').convert('RGB')
    meta = {
        'patient_id':    str(getattr(ds, 'PatientID', 'unknown')),
        'modality':      str(getattr(ds, 'Modality', 'unknown')),
        'body_part':     str(getattr(ds, 'BodyPartExamined', 'unknown')),
        'rows':          int(pixels.shape[0]),
        'columns':       int(pixels.shape[1]),
        'pixel_spacing': list(getattr(ds, 'PixelSpacing', [1.0, 1.0])),
    }
    return img, meta

if not Path(DICOM_PATH).exists():
    print('No DICOM found — using synthetic chest X-ray phantom')
    rng = np.random.default_rng(42)
    H, W = 512, 512
    arr = np.zeros((H, W), dtype=np.float32) + 0.1
    cx = W // 2
    for x in range(W):
        arr[:, x] += np.exp(-0.5 * ((x - cx) / 60)**2) * 0.7
    Y, X = np.mgrid[0:H, 0:W]
    for yc, xc, ry, rx in [(H//2, cx-100, 160, 85), (H//2, cx+100, 160, 85)]:
        arr[((X-xc)/rx)**2 + ((Y-yc)/ry)**2 < 1] -= 0.25
    arr = (arr + 0.02 * rng.standard_normal((H, W))).clip(0, 1)
    pil_img = Image.fromarray((arr * 255).astype(np.uint8), 'L').convert('RGB')
    meta = {'modality': 'CXR_SYNTHETIC', 'rows': H, 'columns': W, 'patient_id': 'synthetic'}
else:
    pil_img, meta = load_dicom_as_pil(DICOM_PATH)

print(f'Image: {pil_img.size}  modality: {meta["modality"]}')
pil_img

## 2. Load MedGemma (with optional 4-bit quantization)

In [ ]:
import torch
from transformers import AutoProcessor, AutoModelForImageTextToText, BitsAndBytesConfig

print(f'GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"}')
print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB' if torch.cuda.is_available() else '')

processor = AutoProcessor.from_pretrained(MODEL_ID, token=HF_TOKEN)

if QUANTIZE_4BIT:
    # 4-bit NF4 quantization via bitsandbytes — fits in ~3.5GB VRAM
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type='nf4',
        bnb_4bit_compute_dtype=torch.bfloat16,
        bnb_4bit_use_double_quant=True,   # nested quantization saves ~0.4 bpw
    )
    model = AutoModelForImageTextToText.from_pretrained(
        MODEL_ID, token=HF_TOKEN,
        quantization_config=bnb_config,
        device_map='auto',
    )
    print('Loaded in 4-bit NF4 quantization')
else:
    # Full bfloat16 — needs ~8GB VRAM
    model = AutoModelForImageTextToText.from_pretrained(
        MODEL_ID, token=HF_TOKEN,
        torch_dtype=torch.bfloat16,
        device_map='auto',
    )
    print('Loaded in bfloat16')

model.eval()
n_params = sum(p.numel() for p in model.parameters())
print(f'Parameters: {n_params:,}  ({n_params/1e9:.2f}B)')

## 3. Register activation hooks

In [ ]:
captured = {}
hooks    = []

def make_hook(name):
    def fn(module, inp, out):
        captured[name] = {
            'input':  inp[0].detach().float().cpu().numpy(),
            'output': out.detach().float().cpu().numpy(),
        }
    return fn

for layer_idx in range(TARGET_LAYERS):
    try:
        lm = model.language_model.model.layers[layer_idx]
        for pname in ('q_proj', 'k_proj', 'v_proj', 'o_proj'):
            proj = getattr(lm.self_attn, pname)
            key  = f'lm.layer{layer_idx}.attn.{pname}'
            hooks.append(proj.register_forward_hook(make_hook(key)))
        for pname in ('gate_proj', 'up_proj', 'down_proj'):
            if hasattr(lm.mlp, pname):
                proj = getattr(lm.mlp, pname)
                key  = f'lm.layer{layer_idx}.mlp.{pname}'
                hooks.append(proj.register_forward_hook(make_hook(key)))
    except AttributeError as e:
        print(f'Hook warning layer {layer_idx}: {e}')

print(f'Registered {len(hooks)} hooks across {TARGET_LAYERS} transformer layers')

## 4. Run MedGemma forward pass

In [ ]:
QUERY = 'Describe any abnormalities visible in this medical image.'

messages = [{'role': 'user', 'content': [
    {'type': 'image', 'image': pil_img},
    {'type': 'text',  'text':  QUERY},
]}]

inputs = processor.apply_chat_template(
    messages, add_generation_prompt=True, tokenize=True,
    return_dict=True, return_tensors='pt',
).to(model.device)

print(f'Input tokens: {inputs["input_ids"].shape}')

with torch.no_grad():
    output_ids = model.generate(**inputs, max_new_tokens=256, do_sample=False)

for h in hooks:
    h.remove()

generated = processor.decode(output_ids[0, inputs['input_ids'].shape[1]:], skip_special_tokens=True)

print('\n' + '='*70)
print('MedGemma Response:')
print('='*70)
print(generated)
print('='*70)
print(f'\nCaptured activations from {len(captured)} projections')

## 5. Extract weights (with 4-bit dequantization if needed)

In [ ]:
def extract_weight(layer_idx: int, proj_name: str, from_mlp: bool = False) -> np.ndarray:
    """
    Extract a weight matrix as float32 numpy array.
    Handles both full-precision and 4-bit quantized models.
    """
    lm = model.language_model.model.layers[layer_idx]
    proj = getattr(lm.mlp if from_mlp else lm.self_attn, proj_name)

    if QUANTIZE_4BIT:
        # Dequantize 4-bit weights back to float32 for SVD
        import bitsandbytes as bnb
        if hasattr(proj, 'weight') and hasattr(proj.weight, 'quant_state'):
            W = bnb.functional.dequantize_4bit(
                proj.weight.data,
                proj.weight.quant_state,
            ).float().cpu().numpy()
        else:
            W = proj.weight.detach().float().cpu().numpy()
    else:
        W = proj.weight.detach().float().cpu().numpy()

    return W

W_q = extract_weight(0, 'q_proj')
W_k = extract_weight(0, 'k_proj')
W_v = extract_weight(0, 'v_proj')
W_o = extract_weight(0, 'o_proj')

print('Layer 0 weights extracted:')
for name, W in [('q_proj', W_q), ('k_proj', W_k), ('v_proj', W_v), ('o_proj', W_o)]:
    print(f'  {name}: {W.shape}  dtype={W.dtype}  norm={np.linalg.norm(W):.2f}')

## 6. Save activations

In [ ]:
activation_manifest = []
for layer_name, data in captured.items():
    inp, out = data['input'], data['output']   # [1, seq_len, dim]
    safe = layer_name.replace('.', '_')
    np.save(OUTPUT_DIR / f'input_{safe}.npy',  inp[0])
    np.save(OUTPUT_DIR / f'output_{safe}.npy', out[0])
    activation_manifest.append({
        'layer_name': layer_name, 'input_shape': list(inp[0].shape), 'output_shape': list(out[0].shape),
    })
    print(f'  {layer_name:45s}  in={inp[0].shape}  out={out[0].shape}')

(OUTPUT_DIR / 'activation_manifest.json').write_text(json.dumps({'layers': activation_manifest}, indent=2))
print(f'Saved {len(activation_manifest)} activation pairs')

## 7. Photonic simulation (inline Clements)

These functions are extracted from PhotoMedGemma's `src/compiler/clements.py`.
They implement the same LEFT-multiply Clements decomposition used in the compiler.

In [ ]:
def null_params(a, b):
    if abs(b) < 1e-15: return 0.0, 0.0
    phi   = float(np.angle(b) - np.angle(a))
    theta = float(2.0 * np.arctan2(abs(b), abs(a)))
    return theta % (2*np.pi), phi % (2*np.pi)

def T(theta, phi):
    return np.array([
        [np.cos(theta/2), 1j*np.exp(-1j*phi)*np.sin(theta/2)],
        [1j*np.exp(1j*phi)*np.sin(theta/2), np.cos(theta/2)]
    ])

def clements_decompose(U):
    N = U.shape[0]
    W = U.astype(complex).copy()
    mzis = []
    for col in range(N-1):
        for row in range(N-1, col, -1):
            theta, phi = null_params(W[row-1, col], W[row, col])
            Td = T(theta, phi).conj().T
            rows = W[[row-1, row], :].copy()
            W[[row-1, row], :] = Td @ rows
            mzis.append((row-1, col, theta, phi))
    return mzis, np.angle(np.diag(W))

def photonic_forward(mzis, phase_screen, x):
    field = x.astype(complex).copy()
    field *= np.exp(1j * phase_screen)        # phase screen D
    for (i, col, theta, phi) in reversed(mzis):
        f_ij = field[[i, i+1]].copy()
        field[[i, i+1]] = T(theta, phi) @ f_ij
    return field

# Sanity check
_m, _ps = clements_decompose(np.eye(4, dtype=complex))
_err = np.linalg.norm(photonic_forward(_m, _ps, np.ones(4)) - 1)
print(f'Clements sanity check: error = {_err:.2e} (should be ~0)')

In [ ]:
# ── Compile Q projection on real MedGemma weights ──────────────────────────
from scipy.linalg import svd as scipy_svd

print(f'SVD-compiling Q projection  shape={W_q.shape}  rank={SVD_RANK}...')
U_sq, s, Vh_sq = scipy_svd(W_q.astype(np.float64), full_matrices=True)
r = min(SVD_RANK, len(s))
energy = (s[:r]**2).sum() / (s**2).sum()
sigma_n = s[:r] / s[0]    # normalize for optical attenuators

print(f'Rank={r}, energy retained={energy:.4f}')
print(f'Decomposing U ({U_sq.shape[0]}×{U_sq.shape[0]})...')
mzis_U,  ps_U  = clements_decompose(U_sq)
print(f'Decomposing Vh ({Vh_sq.shape[0]}×{Vh_sq.shape[0]})...')
mzis_Vh, ps_Vh = clements_decompose(Vh_sq)
print(f'Done — U: {len(mzis_U):,} MZIs  Vh: {len(mzis_Vh):,} MZIs  total: {len(mzis_U)+len(mzis_Vh):,}')

## 8. Compare photonic vs. digital on real MedGemma activations

In [ ]:
q_key      = 'lm_layer0_attn_q_proj'
q_inp_path = OUTPUT_DIR / f'input_{q_key}.npy'

assert q_inp_path.exists(), f'Run sections 3-6 first. Missing: {q_inp_path}'

q_inputs = np.load(q_inp_path)    # [seq_len, hidden_dim]
N_U, N_Vh = W_q.shape
n_test = min(32, q_inputs.shape[0])

errors = []
for tok_idx in range(n_test):
    x = q_inputs[tok_idx].astype(np.float64)    # real token activation from MedGemma

    # Digital reference (exact full-rank)
    y_digital = W_q.astype(np.float64) @ x

    # SVD approximation (rank-r, what the photonic chip stores)
    y_svd = (U_sq[:, :r] * sigma_n * s[0]) @ (Vh_sq[:r, :] @ x)

    # Photonic simulation (MZI mesh)
    y_vh    = photonic_forward(mzis_Vh, ps_Vh, x)[:r]
    y_sigma = sigma_n * s[0] * y_vh
    field_u = np.zeros(N_U, dtype=complex)
    field_u[:r] = y_sigma
    y_phot  = photonic_forward(mzis_U, ps_U, field_u)

    nd, ns = np.linalg.norm(y_digital)+1e-15, np.linalg.norm(y_svd)+1e-15
    errors.append({
        'tok': tok_idx,
        'svd_vs_digital':      float(np.linalg.norm(y_svd - y_digital) / nd),
        'photonic_vs_svd':     float(np.linalg.norm(y_phot.real - y_svd) / ns),
        'photonic_vs_digital': float(np.linalg.norm(y_phot.real - y_digital) / nd),
    })

mean_svd_err  = np.mean([e['svd_vs_digital']  for e in errors])
mean_phot_err = np.mean([e['photonic_vs_svd'] for e in errors])

print(f'Q projection comparison ({n_test} real MedGemma token activations):')
print(f'{"Token":>5} {"SVD vs Digital":>16} {"Photonic vs SVD":>16} {"Photonic vs Digital":>20}')
print('-'*62)
for e in errors:
    print(f"{e['tok']:>5} {e['svd_vs_digital']:>16.4e} {e['photonic_vs_svd']:>16.4e} {e['photonic_vs_digital']:>20.4e}")

print(f'\nMean SVD vs digital:    {mean_svd_err:.3e}  (rank-{r} truncation of {min(N_U,N_Vh)} singular values)')
print(f'Mean photonic vs SVD:   {mean_phot_err:.3e}  (MZI mesh fidelity — should be ~machine epsilon)')
print(f'\nVerdict: {"PASS" if mean_phot_err < 0.01 else "FAIL"} — photonic chip error < 1%' if mean_phot_err < 0.01 else f'\nVerdict: HIGH ERROR — check decomposition')

## 9. Save all results + weights

In [ ]:
# Save results JSON
results = {
    'model_id':    MODEL_ID,
    'quantized':   QUANTIZE_4BIT,
    'query':       QUERY,
    'response':    generated,
    'dicom_meta':  meta,
    'svd_rank':    SVD_RANK,
    'layer_0_q_proj': {
        'weight_shape':         list(W_q.shape),
        'effective_rank':       r,
        'energy_retained':      float(energy),
        'n_mzis_U':             len(mzis_U),
        'n_mzis_Vh':            len(mzis_Vh),
        'token_comparisons':    errors,
        'mean_svd_vs_digital':  float(mean_svd_err),
        'mean_photonic_vs_svd': float(mean_phot_err),
    }
}
(OUTPUT_DIR / 'photomedgemma_comparison.json').write_text(json.dumps(results, indent=2))

# Save raw weights (for analyze_kaggle_results.py)
for name, W in [('W_q', W_q), ('W_k', W_k), ('W_v', W_v), ('W_o', W_o)]:
    np.save(OUTPUT_DIR / f'{name}_layer0.npy', W)

print('Files saved to output tab:')
for f in sorted(OUTPUT_DIR.iterdir()):
    print(f'  {f.name:55s}  {f.stat().st_size/1024:6.0f} KB')

## 10. Final summary

In [ ]:
print('='*70)
print('PhotoMedGemma vs MedGemma — Validation Summary')
print('='*70)
print(f'  Model:              {MODEL_ID}')
print(f'  Quantization:       {"4-bit NF4" if QUANTIZE_4BIT else "bfloat16"}')
print(f'  Image:              {meta["modality"]}  {meta["rows"]}×{meta["columns"]} px')
print(f'  SVD rank:           {r} / {min(*W_q.shape)}')
print(f'  Energy retained:    {energy:.4f} ({energy*100:.2f}%)')
print(f'  Total MZIs (Q):     {len(mzis_U)+len(mzis_Vh):,}')
print()
print('MedGemma response:')
for line in generated.split("\n")[:6]:
    print(f'  {line}')
print()
print(f'  SVD vs digital error:   {mean_svd_err:.3e}  (rank truncation, inherent trade-off)')
print(f'  Photonic vs SVD error:  {mean_phot_err:.3e}  (MZI mesh — near machine precision)')
print()
print('Next steps:')
print('  1. Download output/photomedgemma_comparison.json')
print('  2. Download output/W_*_layer0.npy')
print('  3. Run locally: python3 scripts/analyze_kaggle_results.py')
print('     --results output/simulations/kaggle_comparison.json')
print('     --weights-dir output/simulations/')
print('='*70)